In [174]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [175]:
import pandas as pd
import numpy as np

In [176]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [177]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl

df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")

df[['Network ID', 'Network Name']].drop_duplicates()



,Network ID,Network Name
0,10,-> BC AGRIWeather
48,2,-> BC-TRAN
49,12,-> BC-FWx
57,12,-> BC-TS
62,<NA>,-> MVRD-AIR
63,5,-> BCH-GSO-HMP
65,14,-> BC-Snow
83,<NA>,-> MVRD-WS
98,<NA>,-> BC-FERN
130,<NA>,-> PC-FWx


### Fix problem

In [178]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_attribution = df[df["ISSUE"].str.contains("Attribution", na=False)]

len(df_attribution)

df_attribution =  df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df_attribution["Network ID"] = pd.to_numeric(df_attribution["Network ID"], errors="coerce").astype("Int64")
df_attribution["Station ID"] = pd.to_numeric(df_attribution["Station ID"], errors="coerce").astype("Int64")
df_attribution


,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,Attribution,2322,12,BC-TS,945,TS NAKA CREEK,"115.2144 W, 49.6673 N, Elev. 1051 m -> 126.433..."
1,Attribution,2633,9,BC-Air -> MVRD-AIR,E246240 -> T034,Abbotsford Airport - Walmsley Road,"122.343 W, 49.0235 N, Elev. 65 m"
2,Attribution,12610,9,BC-Air -> MVRD-AIR,New Westminster Sapperton Park -> T046,NA -> New Westminister,"122.894487 W, 49.227045 N, Elev. null m -> 122..."
3,Attribution,12608,9,BC-Air -> MVRD-AIR,Surrey East -> T015,NA -> Surrey East,"122.6942 W, 49.1328 N, Elev. null m -> 122.695..."
4,Attribution,12529,9,BC-Air -> MVRD-AIR,Abbotsford A Columbia Street -> T045,NA -> Abbotsford Airport,"122.3266 W, 49.0215 N, Elev. null m -> 122.326..."
...,...,...,...,...,...,...,...
97,"Location, Attribution",3301,12,BC-TS,1066,TS MCNABB,"123.38722 W, 49.5855 N, Elev. 154 m -> 123.387..."
98,"Location, Attribution",12453,12,BC-TS,1362,TS BURMAN,"126.032697 W, 49.610222 N, Elev. 77 m -> 126.0..."
99,"Name, Location, Attribution",12458,12,BC-TS,1378,TS ATLUCK,"127.0796111 W, 50.2678056 N, Elev. 462 m -> 12..."
100,"Name, Location, Attribution",12526,12,BC-TS,1398,TS NAHMINT,"125.1218 W, 49.2074 N, Elev. 341 m -> 125.1218..."


### update native_id

In [40]:
df_attribution_id = df_attribution[['ISSUE','Station ID', 'Network ID', 'Native ID',]]

split_ids = df_attribution_id['Native ID'].astype(str).str.split('->', expand=True)

df_attribution_id['old_native_id'] = split_ids[0].str.strip()
df_attribution_id['new_native_id'] = split_ids[1].str.strip()


df_id_update = df_attribution_id[df_attribution_id['new_native_id'].notna()]

df_id_update_filtered = df_id_update[
    df_attribution_id['new_native_id'].notna() &
    ~df_attribution_id['ISSUE'].str.contains("Concatenate", na=False) &
    ~df_attribution_id['ISSUE'].str.contains("Delete", na=False)
]

len(df_id_update_filtered)
df_id_update_filtered

/tmp/ipykernel_86356/4235452717.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_id['old_native_id'] = split_ids[0].str.strip()
/tmp/ipykernel_86356/4235452717.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_id['new_native_id'] = split_ids[1].str.strip()
/tmp/ipykernel_86356/4235452717.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_id_update_filtered = df_id_update[


,ISSUE,Station ID,Network ID,Native ID,old_native_id,new_native_id
1,Attribution,2633,9,E246240 -> T034,E246240,T034
2,Attribution,12610,9,New Westminster Sapperton Park -> T046,New Westminster Sapperton Park,T046
3,Attribution,12608,9,Surrey East -> T015,Surrey East,T015
4,Attribution,12529,9,Abbotsford A Columbia Street -> T045,Abbotsford A Columbia Street,T045
5,Attribution,12538,9,Coquitlam Douglas College -> T032,Coquitlam Douglas College,T032
6,Attribution,12602,9,North Delta -> T013,North Delta,T013
7,Attribution,12607,9,Richmond South -> T017,Richmond South,T017
8,Attribution,12583,9,Vancouver International Airport #2 -> T031,Vancouver International Airport #2,T031
21,Attribution,2565,14,4A02P -> PYN,4A02P,PYN
36,"Attribution, ID",12951,9,E304550 ->85A,E304550,85A


In [41]:

check_sql = sa.text("""
SELECT station_id, native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.connect() as conn:
    for _, row in df_id_update_filtered.iterrows():
        res = conn.execute(
            check_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_native_id = res.native_id

        if db_native_id == row["old_native_id"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_native_id} → {row['new_native_id']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_native_id}, expected={row['old_native_id']}"
            )

✅ OK to update station 2633: E246240 → T034
✅ OK to update station 12610: New Westminster Sapperton Park → T046
✅ OK to update station 12608: Surrey East → T015
✅ OK to update station 12529: Abbotsford A Columbia Street → T045
✅ OK to update station 12538: Coquitlam Douglas College → T032
✅ OK to update station 12602: North Delta → T013
✅ OK to update station 12607: Richmond South → T017
✅ OK to update station 12583: Vancouver International Airport #2 → T031
✅ OK to update station 2565: 4A02P → PYN
✅ OK to update station 12951: E304550 → 85A
✅ OK to update station 12960: 1 → C2B10A78
✅ OK to update station 11433: 1239 → CBA24550
✅ OK to update station 11431: 1240 → CBA47004
✅ OK to update station 11430: 1241 → CBA4A66C
✅ OK to update station 11432: 1242 → CBA48080
✅ OK to update station 12037: 1260 → CBA572FE
✅ OK to update station 12036: 1261 → CBA1E2A6
✅ OK to update station 12051: 1262 → CBA18740
✅ OK to update station 12038: 1263 → CBA1F1D0
✅ OK to update station 12039: 1264 → CBA1

In [42]:
update_station_sql = sa.text("""
UPDATE meta_station
SET native_id = :new_native_id
WHERE station_id = :station_id
  AND native_id = :old_native_id
""")

with engine.begin() as conn:
    for _, row in df_id_update_filtered.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "station_id": int(row["Station ID"]),
                "old_native_id": row["old_native_id"],
                "new_native_id": row["new_native_id"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['Station ID']}: "
                f"{row['old_native_id']} → {row['new_native_id']}"
            )
        else:
            # check why it failed
            res = conn.execute(
                check_sql,
                {"station_id": int(row["Station ID"])}
            ).fetchone()

            if res is None:
                print(f"❌ Station {row['Station ID']} not found")
            else:
                print(
                    f"⚠️ Skipped station {row['Station ID']}: "
                    f"DB={res.native_id}, expected={row['old_native_id']}"
                )

✅ Updated station 2633: E246240 → T034
✅ Updated station 12610: New Westminster Sapperton Park → T046
✅ Updated station 12608: Surrey East → T015
✅ Updated station 12529: Abbotsford A Columbia Street → T045
✅ Updated station 12538: Coquitlam Douglas College → T032
✅ Updated station 12602: North Delta → T013
✅ Updated station 12607: Richmond South → T017
✅ Updated station 12583: Vancouver International Airport #2 → T031
✅ Updated station 2565: 4A02P → PYN
✅ Updated station 12951: E304550 → 85A
✅ Updated station 12960: 1 → C2B10A78
✅ Updated station 11433: 1239 → CBA24550
✅ Updated station 11431: 1240 → CBA47004
✅ Updated station 11430: 1241 → CBA4A66C
✅ Updated station 11432: 1242 → CBA48080
✅ Updated station 12037: 1260 → CBA572FE
✅ Updated station 12036: 1261 → CBA1E2A6
✅ Updated station 12051: 1262 → CBA18740
✅ Updated station 12038: 1263 → CBA1F1D0
✅ Updated station 12039: 1264 → CBA19436
✅ Updated station 12042: 1265 → CBA1B2DA
✅ Updated station 12089: 0310177 → T004
✅ Updated stat

In [43]:
verify_sql = sa.text("""
SELECT native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_id_update_filtered.iterrows():
        # verify
        res = conn.execute(
            verify_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res and res.native_id == row["new_native_id"]:
            print(
                f"✅ Verified: station {row['Station ID']} "
                f"native_id = {res.native_id}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {row['Station ID']}: "
                f"DB={res.native_id if res else None}, "
                f"expected={row['new_native_id']}"
            )

✅ Verified: station 2633 native_id = T034
✅ Verified: station 12610 native_id = T046
✅ Verified: station 12608 native_id = T015
✅ Verified: station 12529 native_id = T045
✅ Verified: station 12538 native_id = T032
✅ Verified: station 12602 native_id = T013
✅ Verified: station 12607 native_id = T017
✅ Verified: station 12583 native_id = T031
✅ Verified: station 2565 native_id = PYN
✅ Verified: station 12951 native_id = 85A
✅ Verified: station 12960 native_id = C2B10A78
✅ Verified: station 11433 native_id = CBA24550
✅ Verified: station 11431 native_id = CBA47004
✅ Verified: station 11430 native_id = CBA4A66C
✅ Verified: station 11432 native_id = CBA48080
✅ Verified: station 12037 native_id = CBA572FE
✅ Verified: station 12036 native_id = CBA1E2A6
✅ Verified: station 12051 native_id = CBA18740
✅ Verified: station 12038 native_id = CBA1F1D0
✅ Verified: station 12039 native_id = CBA19436
✅ Verified: station 12042 native_id = CBA1B2DA
✅ Verified: station 12089 native_id = T004
✅ Verified: st

### update station name

In [51]:
df_attribution_name = df_attribution[['ISSUE','Station ID', 'Network ID', 'Unique Names',]]
df_attribution_name

def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_attribution_name[['old_name', 'new_name', 'n_names']] = (
    df_attribution_name['Unique Names'].apply(split_station_name)
)

df_attribution_name = df_attribution_name.drop(columns= 'Unique Names')
df_attribution_name[0:30]
# split_ids = df_attribution_id['Native ID'].astype(str).str.split('->', expand=True)

# df_attribution_id['old_native_id'] = split_ids[0].str.strip()
# df_attribution_id['new_native_id'] = split_ids[1].str.strip()


# df_id_update = df_attribution_id[df_attribution_id['new_native_id'].notna()]

# df_id_update_filtered = df_id_update[
#     df_attribution_id['new_native_id'].notna() &
#     ~df_attribution_id['ISSUE'].str.contains("Concatenate", na=False) &
#     ~df_attribution_id['ISSUE'].str.contains("Delete", na=False)
# ]

# len(df_id_update_filtered)
# df_id_update_filtered

/tmp/ipykernel_86356/4002969949.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_name[['old_name', 'new_name', 'n_names']] = (
/tmp/ipykernel_86356/4002969949.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_name[['old_name', 'new_name', 'n_names']] = (


,ISSUE,Station ID,Network ID,old_name,new_name,n_names
0,Attribution,2322,12,TS NAKA CREEK,TS NAKA CREEK,1.0
1,Attribution,2633,9,Abbotsford Airport - Walmsley Road,Abbotsford Airport - Walmsley Road,1.0
2,Attribution,12610,9,NA,New Westminister,2.0
3,Attribution,12608,9,NA,Surrey East,2.0
4,Attribution,12529,9,NA,Abbotsford Airport,2.0
5,Attribution,12538,9,NA,Coquitlam,2.0
6,Attribution,12602,9,NA,North Delta,2.0
7,Attribution,12607,9,NA,Richmond South,2.0
8,Attribution,12583,9,NA,YVR,2.0
9,Attribution,2632,9,South Langley Met,South Langley Met,1.0


In [58]:
df_name_update = df_attribution_name[df_attribution_name['old_name'] != df_attribution_name['new_name'] ]

df_name_update = df_attribution_name[
    (df_attribution_name['old_name'] != df_attribution_name['new_name']) &
    (~df_attribution_name['ISSUE'].str.contains("delete", na=False))
]

df_name_update

,ISSUE,Station ID,Network ID,old_name,new_name,n_names
2,Attribution,12610,9,NA,New Westminister,2.0
3,Attribution,12608,9,NA,Surrey East,2.0
4,Attribution,12529,9,NA,Abbotsford Airport,2.0
5,Attribution,12538,9,NA,Coquitlam,2.0
6,Attribution,12602,9,NA,North Delta,2.0
7,Attribution,12607,9,NA,Richmond South,2.0
8,Attribution,12583,9,NA,YVR,2.0
23,"Attribution, Concatenate",12559,9,NA,Peace Valley Attachie Flat Upper Terrace,2.0
33,"Attribution, Delete Reference to 12601",12204,9,North Burnaby Capitol Hill,Burnaby-Capitol Hill,2.0
34,"Attribution, Delete Reference to 12603",12207,9,North Vancouver Mahon Park,N. Vancouver-Mahon Park,2.0


In [59]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name
FROM meta_history
WHERE station_id = :station_id
""")

import numpy as np

with engine.begin() as conn:
    for _, row in df_name_update.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name}, the names match "
            )


✅ Station 12610, NA, the names match 
✅ Station 12608, NA, the names match 
✅ Station 12529, NA, the names match 
✅ Station 12538, NA, the names match 
✅ Station 12602, NA, the names match 
✅ Station 12607, NA, the names match 
✅ Station 12583, NA, the names match 
✅ Station 12559, NA, the names match 
✅ Station 12204, North Burnaby Capitol Hill, the names match 
✅ Station 12207, North Vancouver Mahon Park, the names match 
⚠️ Station 12951 (DB, XLS values differ)
   DB : station name is None
   XLS: station name is Fort St John 85 Avenue
⚠️ Station 12960 (DB, XLS values differ)
   DB : station name is None
   XLS: station name is 
✅ Station 12232, Port Moody Rocky Point Park, the names match 
⚠️ Station 13011 (DB, XLS values differ)
   DB : station name is None
   XLS: station name is 
⚠️ Station 13012 (DB, XLS values differ)
   DB : station name is None
   XLS: station name is 
⚠️ Station 13013 (DB, XLS values differ)
   DB : station name is None
   XLS: station name is 
⚠️ Station 1

In [60]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_name_update.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}) → "
                f"({new_name})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 12610: (NA) → (New Westminister)
✅ Updated station 12608: (NA) → (Surrey East)
✅ Updated station 12529: (NA) → (Abbotsford Airport)
✅ Updated station 12538: (NA) → (Coquitlam)
✅ Updated station 12602: (NA) → (North Delta)
✅ Updated station 12607: (NA) → (Richmond South)
✅ Updated station 12583: (NA) → (YVR)
✅ Updated station 12559: (NA) → (Peace Valley Attachie Flat Upper Terrace)
✅ Updated station 12204: (North Burnaby Capitol Hill) → (Burnaby-Capitol Hill)
✅ Updated station 12207: (North Vancouver Mahon Park) → (N. Vancouver-Mahon Park)
✅ Updated station 12951: (Fort St John 85 Avenue) → (85th Avenue)
✅ Updated station 12960: () → (TOFINO PCA)
✅ Updated station 12232: (Port Moody Rocky Point Park) → (Port Moody)
✅ Updated station 13011: () → (Capilano Fire Weather)
✅ Updated station 13012: () → (Seymour Fire Weather)
✅ Updated station 13013: () → (Coquitlam Fire Weather)
✅ Updated station 13014: () → (Port Edwards Sunset Drive)
✅ Updated station 2647: (Pitt Meadows)

### Update location

In [179]:
df_attribution_loc = df_attribution[['ISSUE','Station ID', 'Network ID', 'Unique Location (String)',]]

df_loc = df_attribution_loc.loc[df_attribution_loc['Unique Location (String)'].astype(str).str.contains('->', na=False)]


df_loc

,ISSUE,Station ID,Network ID,Unique Location (String)
0,Attribution,2322,12,"115.2144 W, 49.6673 N, Elev. 1051 m -> 126.433..."
2,Attribution,12610,9,"122.894487 W, 49.227045 N, Elev. null m -> 122..."
3,Attribution,12608,9,"122.6942 W, 49.1328 N, Elev. null m -> 122.695..."
4,Attribution,12529,9,"122.3266 W, 49.0215 N, Elev. null m -> 122.326..."
5,Attribution,12538,9,"122.7914 W, 49.2881 N, Elev. null m -> 122.791..."
6,Attribution,12602,9,"122.9017 W, 49.1583 N, Elev. null m -> 122.901..."
7,Attribution,12607,9,"123.1083 W, 49.1414 N, Elev. null m -> 123.108..."
8,Attribution,12583,9,"123.1522 W, 49.1864 N, Elev. null m -> 123.152..."
23,"Attribution, Concatenate",12559,9,"121.41944 W, 56.231213 N, Elev. null m -> 121...."
26,"Attribution, Concatenate and delete",12563,9,"122.8492 W, 49.2808 N, Elev. null m -> 122.849..."


In [180]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_loc[cols] = df_loc['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)
# 
df_loc = df_loc.drop(columns=['Unique Location (String)'])


df_loc = df_loc.dropna(
    subset=['new_lat', 'new_lon', 'new_elev'],
    how='all'
)

df_loc


/tmp/ipykernel_86356/64680190.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_loc[cols] = df_loc['Unique Location (String)'].apply(
/tmp/ipykernel_86356/64680190.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_loc[cols] = df_loc['Unique Location (String)'].apply(
/tmp/ipykernel_86356/64680190.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

,ISSUE,Station ID,Network ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
0,Attribution,2322,12,49.667300,-115.214400,1051.0,50.370100,-126.433900,515.0
2,Attribution,12610,9,49.227045,-122.894487,NaN,49.227045,-122.894487,45.0
3,Attribution,12608,9,49.132800,-122.694200,NaN,49.134235,-122.695491,79.0
4,Attribution,12529,9,49.021500,-122.326600,NaN,49.021500,-122.326500,65.0
5,Attribution,12538,9,49.288100,-122.791400,NaN,49.288300,-122.791600,38.0
6,Attribution,12602,9,49.158300,-122.901700,NaN,49.158300,-122.901700,113.0
7,Attribution,12607,9,49.141400,-123.108300,NaN,49.141400,-123.108200,4.0
8,Attribution,12583,9,49.186400,-123.152200,NaN,49.186300,-123.152400,1.0
23,"Attribution, Concatenate",12559,9,56.231213,-121.419440,NaN,56.231213,-121.419440,480.0
26,"Attribution, Concatenate and delete",12563,9,49.280800,-122.849200,NaN,49.280900,-122.849300,13.0


In [77]:
check_sql = sa.text("""
SELECT
    station_id,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_loc.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ):
            print(
                f"⚠️ Station {station_id} skipped (DB values differ)\n"
                f"   DB : lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )
            continue


⚠️ Station 2322 skipped (DB values differ)
   DB : lat=50.3701, lon=-126.4339, elev=515.0
   XLS: lat=49.6673, lon=-115.2144, elev=1051.0
⚠️ Station 13043 skipped (DB values differ)
   DB : lat=49.6233, lon=-117.0436, elev=815
   XLS: lat=49.62333333, lon=-117.0436111, elev=830.0
⚠️ Station 13044 skipped (DB values differ)
   DB : lat=49.6445, lon=-117.0626, elev=1270
   XLS: lat=49.64444444, lon=-117.0625, elev=1290.0
⚠️ Station 13045 skipped (DB values differ)
   DB : lat=49.6693, lon=-117.0643, elev=1715
   XLS: lat=49.66916667, lon=-117.0641667, elev=1735.0
⚠️ Station 13046 skipped (DB values differ)
   DB : lat=49.69, lon=-117.0865, elev=2085
   XLS: lat=49.69, lon=-117.0863889, elev=2045.0
⚠️ Station 2321 skipped (DB values differ)
   DB : lat=48.5713, lon=-124.2, elev=294.0
   XLS: lat=48.5711, lon=-124.2006, elev=296.0
⚠️ Station 2338 skipped (DB values differ)
   DB : lat=50.13242, lon=-126.92823, elev=120.0
   XLS: lat=50.13242, lon=-126.92823, elev=130.0
⚠️ Station 2339 skip

In [181]:
import numpy as np

update_sql = sa.text("""
UPDATE meta_history
SET
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_loc.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_lat}, {old_lon}, {old_elev}) → "
                f"({new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 2322: (49.6673, -115.2144, 1051.0) → (50.3701, -126.4339, 515.0)
✅ Updated station 12610: (49.227045, -122.894487, nan) → (49.227045, -122.894487, 45.0)
✅ Updated station 12608: (49.1328, -122.6942, nan) → (49.134235, -122.695491, 79.0)
✅ Updated station 12529: (49.0215, -122.3266, nan) → (49.0215, -122.3265, 65.0)
✅ Updated station 12538: (49.2881, -122.7914, nan) → (49.2883, -122.7916, 38.0)
✅ Updated station 12602: (49.1583, -122.9017, nan) → (49.1583, -122.9017, 113.0)
✅ Updated station 12607: (49.1414, -123.1083, nan) → (49.1414, -123.1082, 4.0)
✅ Updated station 12583: (49.1864, -123.1522, nan) → (49.1863, -123.1524, 1.0)
✅ Updated station 12559: (56.231213, -121.41944, nan) → (56.231213, -121.41944, 480.0)
✅ Updated station 12563: (49.2808, -122.8492, nan) → (49.2809, -122.8493, 13.0)
✅ Updated station 12960: (nan, nan, nan) → (49.0796, -125.7666, 24.0)
✅ Updated station 11433: (51.96803333, -116.7148167, 1442.0) → (51.9681, -116.7145, 1442.0)
✅ Updated station

#### Attribution update

In [93]:
df_attribution = df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name']]
df_attribution

def split_network_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
    df_attribution['Network Name'].apply(split_network_name)
)

df_attribution 

/tmp/ipykernel_86356/508565202.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
/tmp/ipykernel_86356/508565202.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
0,Attribution,2322,12,BC-TS,BC-TS,BC-TS,1
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AIR,2
...,...,...,...,...,...,...,...
97,"Location, Attribution",3301,12,BC-TS,BC-TS,BC-TS,1
98,"Location, Attribution",12453,12,BC-TS,BC-TS,BC-TS,1
99,"Name, Location, Attribution",12458,12,BC-TS,BC-TS,BC-TS,1
100,"Name, Location, Attribution",12526,12,BC-TS,BC-TS,BC-TS,1


In [ ]:
### Check if the new_net_name exist
from sqlalchemy import text

new_net_name = df_attribution['new_net_name'].unique()

exists_sql = text("""
SELECT EXISTS (
    SELECT network_id
    FROM meta_network 
    WHERE network_name = :network_name
)
""")


existing_networks = set()

for name in new_net_name:
    exists = session.execute(
        exists_sql,
        {"network_name": name}
    ).scalar()

    if exists:
        existing_networks.add(name)

existing_networks

In [106]:
df_attribution_2attri = df_attribution[df_attribution['n_net_names'] == 2]

df_attribution_2attri['new_net_name'] = (
    df_attribution_2attri['new_net_name']
    .replace(['MVRD-AIR', 'MVRD'], 'MVRD-AQ')
)

df_attribution_2attri

/tmp/ipykernel_86356/3822543348.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_2attri['new_net_name'] = (


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
5,Attribution,12538,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
...,...,...,...,...,...,...,...
86,"Attribution,Concatenate w/12536",2637,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
87,"Attribution,Delete Reference to 12593",12091,9,BC-Air -> MVRD,BC-Air,MVRD-AQ,2
88,"Attribution,ID",1581,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
89,"Attribution,ID, Delete Reference to 12591",12086,9,BC-Air -> MVRD,BC-Air,MVRD-AQ,2


### select the netwok_id based on new_net_name, summarize back to the table 

In [132]:
# Create an empty column first
df_attribution_2attri['new_network_id'] = None

with engine.connect() as conn:
    for idx, row in df_attribution_2attri.iterrows():
        # Query network_id for this row's new_net_name
        res = conn.execute(
            sa.text("""
                SELECT network_id
                FROM meta_network
                WHERE network_name = :network_name
            """),
            {"network_name": row['new_net_name']}
        ).mappings().fetchone()

        if res is not None:
            df_attribution_2attri.at[idx, 'new_network_id'] = res['network_id']
        else:
            print(f"❌ Network '{row['new_net_name']}' not found for station {row['Station ID']}")

df_attribution_2attri[0:50]

/tmp/ipykernel_86356/1497685280.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_2attri['new_network_id'] = None


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
5,Attribution,12538,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
6,Attribution,12602,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
7,Attribution,12607,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
8,Attribution,12583,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
9,Attribution,2632,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137
10,Attribution,1591,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,137


In [119]:
check_sql = sa.text("""
SELECT
    s.station_id,
    n.network_id,
    n.network_name
FROM meta_network as n
JOIN meta_station as s ON n.network_id = s.network_id
WHERE s.station_id = :station_id
AND n.network_id = :network_id

""")


with engine.connect() as conn:
    for _, row in df_attribution_2attri.iterrows():

        res = conn.execute(
            check_sql,
            {
                "station_id": int(row["Station ID"]),
                "network_id": int(row["Network ID"]),
            }
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_network_name = res[2]

        if db_network_name == row["old_net_name"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_network_name} → {row['new_net_name']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_network_name}, expected={row['Network']}"
            )

✅ OK to update station 2633: BC-Air → MVRD-AQ
✅ OK to update station 12610: BC-Air → MVRD-AQ
✅ OK to update station 12608: BC-Air → MVRD-AQ
✅ OK to update station 12529: BC-Air → MVRD-AQ
✅ OK to update station 12538: BC-Air → MVRD-AQ
✅ OK to update station 12602: BC-Air → MVRD-AQ
✅ OK to update station 12607: BC-Air → MVRD-AQ
✅ OK to update station 12583: BC-Air → MVRD-AQ
✅ OK to update station 2632: BC-Air → MVRD-AQ
✅ OK to update station 1591: BC-Air → MVRD-AQ
✅ OK to update station 1584: BC-Air → MVRD-AQ
✅ OK to update station 2569: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2532: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2551: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2521: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2567: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2522: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2548: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2547: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2541: BC-Snow → BCH-GSO-HMP
✅ OK to update station 2565:

In [ ]:
Just select station_id, then update network_name (new_net_name) and network_id (new_network_id) to the new ones in the form 

In [133]:
update_sql = sa.text("""
UPDATE meta_station
SET network_id = :new_network_id
FROM meta_network AS n
WHERE meta_station.station_id = :station_id
  AND meta_station.network_id = n.network_id
""")

update_network_sql = sa.text("""
UPDATE meta_network AS n
SET network_name = :new_network_name
WHERE n.network_id = :new_network_id
""")

with engine.begin() as conn:  # auto-commit / rollback
    for _, row in df_attribution_2attri.iterrows():

        # 1. Update the network_id in meta_station
        result_station = conn.execute(
            update_sql,
            {
                "station_id": int(row["Station ID"]),
                "new_network_id": int(row["new_network_id"])
            }
        )

        # 2. Update the network_name in meta_network
        result_network = conn.execute(
            update_network_sql,
            {
                "new_network_id": int(row["new_network_id"]),
                "new_network_name": row["new_net_name"]
            }
        )

        # ✅ Print checks
        if result_station.rowcount == 1 and result_network.rowcount == 1:
            print(
                f"✅ Station {row['Station ID']} updated: "
                f"network_id → {row['new_network_id']}, "
                f"network_name → {row['new_net_name']}"
            )
        else:
            print(f"⚠️ Check station {row['Station ID']}, updates may have failed")

            


✅ Station 2633 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12610 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12608 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12529 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12538 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12602 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12607 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 12583 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 2632 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 1591 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 1584 updated: network_id → 137, network_name → MVRD-AQ
✅ Station 2569 updated: network_id → 5, network_name → BCH-GSO-HMP
✅ Station 2532 updated: network_id → 5, network_name → BCH-GSO-HMP
✅ Station 2551 updated: network_id → 5, network_name → BCH-GSO-HMP
✅ Station 2521 updated: network_id → 5, network_name → BCH-GSO-HMP
✅ Station 

In [141]:
check_sql = sa.text("""
SELECT
    s.station_id,
    n.network_id,
    n.network_name
FROM meta_network as n
JOIN meta_station as s ON n.network_id = s.network_id
WHERE s.station_id = :station_id

""")


with engine.connect() as conn:
    for _, row in df_attribution_2attri.iterrows():

        res = conn.execute(
            check_sql,
            {
                "station_id": int(row["Station ID"]),
            }
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue
        db_network_id = res[1]
        db_network_name = res[2]

        if db_network_id == row["new_network_id"] and db_network_name == row["new_net_name"]:

            print(
                f"✅ Verified: station {res.station_id}: "
                f"{db_network_id} → {row['new_network_id']}, "
                f"{db_network_name} → {row['new_net_name']}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {res.station_id}: "
                f"DB={db_network_name}, expected={row['new_net_name']}"
            )

✅ Verified: station 2633: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12610: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12608: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12529: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12538: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12602: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12607: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 12583: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 2632: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 1591: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 1584: 137 → 137, MVRD-AQ → MVRD-AQ
✅ Verified: station 2569: 5 → 5, BCH-GSO-HMP → BCH-GSO-HMP
✅ Verified: station 2532: 5 → 5, BCH-GSO-HMP → BCH-GSO-HMP
✅ Verified: station 2551: 5 → 5, BCH-GSO-HMP → BCH-GSO-HMP
✅ Verified: station 2521: 5 → 5, BCH-GSO-HMP → BCH-GSO-HMP
✅ Verified: station 2567: 5 → 5, BCH-GSO-HMP → BCH-GSO-HMP
✅ Verified: station 2522: 5 → 5, BCH-GSO-HMP → BCH-GSO-HMP
✅ Verified: station 2548: 5 → 5, B

### Only one attributes gives

In [142]:
df_attribution_1attri = df_attribution[df_attribution['n_net_names'] == 1]

df_attribution_1attri['new_net_name'] = (
    df_attribution_1attri['new_net_name']
    .replace(['MVRD-AIR', 'MVRD'], 'MVRD-AQ')
)

df_attribution_1attri

/tmp/ipykernel_86356/2229839121.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_1attri['new_net_name'] = (


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
0,Attribution,2322,12,BC-TS,BC-TS,BC-TS,1
80,"Attribution, Location",13043,11,BC-FERN,BC-FERN,BC-FERN,1
81,"Attribution, Location",13044,11,BC-FERN,BC-FERN,BC-FERN,1
82,"Attribution, Location",13045,11,BC-FERN,BC-FERN,BC-FERN,1
83,"Attribution, Location",13046,11,BC-FERN,BC-FERN,BC-FERN,1
91,"Location, Attribution",2321,12,BC-TS,BC-TS,BC-TS,1
92,"Location, Attribution",2325,12,BC-TS,BC-TS,BC-TS,1
93,"Location, Attribution",2338,12,BC-TS,BC-TS,BC-TS,1
94,"Location, Attribution",2339,12,BC-TS,BC-TS,BC-TS,1
95,"Location, Attribution",2350,12,BC-TS,BC-TS,BC-TS,1


In [149]:
check_sql = sa.text("""
SELECT
    s.station_id,
    n.network_id,
    n.network_name
FROM meta_network as n
JOIN meta_station as s ON n.network_id = s.network_id
WHERE s.station_id = :station_id

""")


with engine.connect() as conn:
    for _, row in df_attribution_1attri.iterrows():

        res = conn.execute(
            check_sql,
            {
                "station_id": int(row["Station ID"]),
            }
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue
        db_network_id = res[1]
        db_network_name = res[2]

        if db_network_name == row["old_net_name"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_network_name} → {row['new_net_name']}, "
                f"DB={db_network_id}, expected={row['Network ID']}"

            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_network_name}, expected={row['old_net_name']}, "
                f"DB={db_network_id}, expected={row['Network ID']}"
            )

⚠️ Mismatch for station 2322: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
✅ OK to update station 13043: BC-FERN → BC-FERN, DB=11, expected=11
✅ OK to update station 13044: BC-FERN → BC-FERN, DB=11, expected=11
✅ OK to update station 13045: BC-FERN → BC-FERN, DB=11, expected=11
✅ OK to update station 13046: BC-FERN → BC-FERN, DB=11, expected=11
⚠️ Mismatch for station 2321: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 2325: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 2338: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 2339: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 2350: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 3300: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 3301: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 12453: DB=BC-FERN, expected=BC-TS, DB=12, expected=12
⚠️ Mismatch for station 12458: DB=

In [150]:
# Create an empty column first
df_attribution_1attri['new_network_id'] = None

with engine.connect() as conn:
    for idx, row in df_attribution_1attri.iterrows():
        # Query network_id for this row's new_net_name
        res = conn.execute(
            sa.text("""
                SELECT network_id
                FROM meta_network
                WHERE network_name = :network_name
            """),
            {"network_name": row['new_net_name']}
        ).mappings().fetchone()

        if res is not None:
            df_attribution_1attri.at[idx, 'new_network_id'] = res['network_id']
        else:
            print(f"❌ Network '{row['new_net_name']}' not found for station {row['Station ID']}")

df_attribution_1attri[0:50]

/tmp/ipykernel_86356/1611397296.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution_1attri['new_network_id'] = None


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id
0,Attribution,2322,12,BC-TS,BC-TS,BC-TS,1,142
80,"Attribution, Location",13043,11,BC-FERN,BC-FERN,BC-FERN,1,11
81,"Attribution, Location",13044,11,BC-FERN,BC-FERN,BC-FERN,1,11
82,"Attribution, Location",13045,11,BC-FERN,BC-FERN,BC-FERN,1,11
83,"Attribution, Location",13046,11,BC-FERN,BC-FERN,BC-FERN,1,11
91,"Location, Attribution",2321,12,BC-TS,BC-TS,BC-TS,1,142
92,"Location, Attribution",2325,12,BC-TS,BC-TS,BC-TS,1,142
93,"Location, Attribution",2338,12,BC-TS,BC-TS,BC-TS,1,142
94,"Location, Attribution",2339,12,BC-TS,BC-TS,BC-TS,1,142
95,"Location, Attribution",2350,12,BC-TS,BC-TS,BC-TS,1,142


In [163]:
check_sql = sa.text("""
SELECT
    s.station_id,
    n.network_id,
    n.network_name
FROM meta_network as n
JOIN meta_station as s ON n.network_id = s.network_id
WHERE s.station_id = :station_id

""")


with engine.connect() as conn:
    for _, row in df_attribution_1attri.iterrows():

        res = conn.execute(
            check_sql,
            {
                "station_id": int(row["Station ID"]),
            }
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_network_id = res[1]
        db_network_name = res[2]

        if db_network_name == row["old_net_name"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_network_name} → {row['new_net_name']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_network_id}, {db_network_name}; "
                f"Change to: {row['new_network_id']}, {row['new_net_name']}"

            )

⚠️ Mismatch for station 2322: DB=12, BC-FERN; Change to: 142, BC-TS
✅ OK to update station 13043: BC-FERN → BC-FERN
✅ OK to update station 13044: BC-FERN → BC-FERN
✅ OK to update station 13045: BC-FERN → BC-FERN
✅ OK to update station 13046: BC-FERN → BC-FERN
⚠️ Mismatch for station 2321: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 2325: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 2338: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 2339: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 2350: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 3300: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 3301: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 12453: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 12458: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 12526: DB=12, BC-FERN; Change to: 142, BC-TS
⚠️ Mismatch for station 12957: DB=12, BC-FERN; Change to:

In [164]:
update_sql = sa.text("""
UPDATE meta_station
SET network_id = :new_network_id
FROM meta_network AS n
WHERE meta_station.station_id = :station_id
  AND meta_station.network_id = n.network_id
""")

update_network_sql = sa.text("""
UPDATE meta_network AS n
SET network_name = :new_network_name
WHERE n.network_id = :new_network_id
""")

with engine.begin() as conn:  # auto-commit / rollback
    for _, row in df_attribution_1attri.iterrows():

        # 1. Update the network_id in meta_station
        result_station = conn.execute(
            update_sql,
            {
                "station_id": int(row["Station ID"]),
                "new_network_id": int(row["new_network_id"])
            }
        )

        # 2. Update the network_name in meta_network
        result_network = conn.execute(
            update_network_sql,
            {
                "new_network_id": int(row["new_network_id"]),
                "new_network_name": row["new_net_name"]
            }
        )

        # ✅ Print checks
        if result_station.rowcount == 1 and result_network.rowcount == 1:
            print(
                f"✅ Station {row['Station ID']} updated: "
                f"network_id → {row['new_network_id']}, "
                f"network_name → {row['new_net_name']}"
            )
        else:
            print(f"⚠️ Check station {row['Station ID']}, updates may have failed")

            


✅ Station 2322 updated: network_id → 142, network_name → BC-TS
✅ Station 13043 updated: network_id → 11, network_name → BC-FERN
✅ Station 13044 updated: network_id → 11, network_name → BC-FERN
✅ Station 13045 updated: network_id → 11, network_name → BC-FERN
✅ Station 13046 updated: network_id → 11, network_name → BC-FERN
✅ Station 2321 updated: network_id → 142, network_name → BC-TS
✅ Station 2325 updated: network_id → 142, network_name → BC-TS
✅ Station 2338 updated: network_id → 142, network_name → BC-TS
✅ Station 2339 updated: network_id → 142, network_name → BC-TS
✅ Station 2350 updated: network_id → 142, network_name → BC-TS
✅ Station 3300 updated: network_id → 142, network_name → BC-TS
✅ Station 3301 updated: network_id → 142, network_name → BC-TS
✅ Station 12453 updated: network_id → 142, network_name → BC-TS
✅ Station 12458 updated: network_id → 142, network_name → BC-TS
✅ Station 12526 updated: network_id → 142, network_name → BC-TS
✅ Station 12957 updated: network_id → 142, n

In [165]:
check_sql = sa.text("""
SELECT
    s.station_id,
    n.network_id,
    n.network_name
FROM meta_network as n
JOIN meta_station as s ON n.network_id = s.network_id
WHERE s.station_id = :station_id

""")


with engine.connect() as conn:
    for _, row in df_attribution_1attri.iterrows():

        res = conn.execute(
            check_sql,
            {
                "station_id": int(row["Station ID"]),
            }
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue
        db_network_id = res[1]
        db_network_name = res[2]

        if db_network_id == row["new_network_id"] and db_network_name == row["new_net_name"]:

            print(
                f"✅ Verified: station {res.station_id}: "
                f"{db_network_id} → {row['new_network_id']}, "
                f"{db_network_name} → {row['new_net_name']}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {res.station_id}: "
                f"DB={db_network_name}, expected={row['new_net_name']}"
            )

✅ Verified: station 2322: 142 → 142, BC-TS → BC-TS
✅ Verified: station 13043: 11 → 11, BC-FERN → BC-FERN
✅ Verified: station 13044: 11 → 11, BC-FERN → BC-FERN
✅ Verified: station 13045: 11 → 11, BC-FERN → BC-FERN
✅ Verified: station 13046: 11 → 11, BC-FERN → BC-FERN
✅ Verified: station 2321: 142 → 142, BC-TS → BC-TS
✅ Verified: station 2325: 142 → 142, BC-TS → BC-TS
✅ Verified: station 2338: 142 → 142, BC-TS → BC-TS
✅ Verified: station 2339: 142 → 142, BC-TS → BC-TS
✅ Verified: station 2350: 142 → 142, BC-TS → BC-TS
✅ Verified: station 3300: 142 → 142, BC-TS → BC-TS
✅ Verified: station 3301: 142 → 142, BC-TS → BC-TS
✅ Verified: station 12453: 142 → 142, BC-TS → BC-TS
✅ Verified: station 12458: 142 → 142, BC-TS → BC-TS
✅ Verified: station 12526: 142 → 142, BC-TS → BC-TS
✅ Verified: station 12957: 142 → 142, BC-TS → BC-TS
